In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

!pip install -q kagglehub segmentation-models-pytorch albumentations timm


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 14.4 MB/s eta 0:00:00


In [ ]:
import shutil
import os

DRIVE_DATASET_DIR = "/content/drive/MyDrive/EchoNet_Colab/dataset"

LOCAL_DATASET_DIR = "/content/local_dataset"

if not os.path.exists(LOCAL_DATASET_DIR):
    print("Copying dataset to local runtime... This will take a moment but saves hours of training time!")
    shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
    print("✅ Copy Done!")

Copying dataset to local runtime... This will take a moment but saves hours of training time!


KeyboardInterrupt: 

In [ ]:
# Define paths in Google Drive
BASE_DIR = "/content/drive/MyDrive/EchoNet_Colab"
RAW_DIR = f"{BASE_DIR}/raw"
DATA_DIR = f"{BASE_DIR}/dataset"
IMG_DIR = "/content/local_dataset/images"
MASK_DIR = "/content/local_dataset/masks"
CKPT_DIR = f"{BASE_DIR}/checkpoints"

# Create directories
for d in [RAW_DIR, IMG_DIR, MASK_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)
print("✅ Directories ready in Google Drive.")


✅ Directories ready in Google Drive.


In [ ]:
import kagglehub
import shutil
import cv2
import pandas as pd
import numpy as np
import glob
from tqdm import tqdm

dataset_path = f"{RAW_DIR}/EchoNet-Dynamic"

# Download if not exists
# if not os.path.exists(dataset_path):
print("Downloading dataset...")
dl_path = kagglehub.dataset_download("mahnurrahman/echonet-dynamic")
shutil.copytree(dl_path, dataset_path, dirs_exist_ok=True)

# Generate dataset (Skip if already generated > 1000 images)
if len(glob.glob(f"{IMG_DIR}/*.png")) < 1000:
    print("Processing videos and generating masks...")
    df = pd.read_csv(f"{dataset_path}/EchoNet-Dynamic/VolumeTracings.csv")
    grouped = df.groupby(["FileName", "Frame"])

    SAVE_SIZE = 256
    counter = 0

    for (fname, frame_num), rows in tqdm(grouped):
        v_path = f"{dataset_path}/EchoNet-Dynamic/Videos/{fname}"
        if not os.path.exists(v_path): continue

        cap = cv2.VideoCapture(v_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        cap.release()

        if not ret: continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        mask = np.zeros(frame.shape, dtype=np.uint8)

        # CORRECTED MASK GENERATION: using both X1,Y1 and X2,Y2
        p1, p2 = [], []
        for _, r in rows.iterrows():
            p1.append([int(r["X1"]), int(r["Y1"])])
            p2.append([int(r["X2"]), int(r["Y2"])])

        points = np.array(p1 + p2[::-1], dtype=np.int32)
        if len(points) > 5:
            cv2.fillPoly(mask, [points], 255)

            frame = cv2.resize(frame, (SAVE_SIZE, SAVE_SIZE))
            mask = cv2.resize(mask, (SAVE_SIZE, SAVE_SIZE))

            cv2.imwrite(f"{IMG_DIR}/{counter}.png", frame)
            cv2.imwrite(f"{MASK_DIR}/{counter}.png", mask)
            counter += 1
print("✅ Dataset and Masks are ready!")


Using Colab cache for faster access to the 'echonet-dynamic' dataset.


KeyboardInterrupt: 

In [ ]:
# ==========================================
# 1. SETUP & PATHS (Drive for Models, Local for Data)
# ==========================================
from google.colab import drive
import os
import kagglehub
import shutil
import cv2
import pandas as pd
import numpy as np
import glob
from tqdm import tqdm

# اتصال درایو فقط برای ذخیره فایل مدل (.pth)
drive.mount('/content/drive')

!pip install -q kagglehub segmentation-models-pytorch albumentations timm

# مسیر ذخیره مدل‌ها در درایو (تا با بسته شدن کولب پاک نشوند)
CKPT_DIR = "/content/drive/MyDrive/EchoNet_Colab/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# 🚀 مسیرهای محلی پرسرعت روی خود سیستم کولب
LOCAL_DIR = "/content/local_data"
RAW_DIR = f"{LOCAL_DIR}/raw"
IMG_DIR = f"{LOCAL_DIR}/images"
MASK_DIR = f"{LOCAL_DIR}/masks"

for d in [RAW_DIR, IMG_DIR, MASK_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Paths ready. Models go to Drive, Data stays Local (Ultra Fast).")

# ==========================================
# 2. DOWNLOAD & GENERATE DATASET LOCALLY
# ==========================================
dataset_path = f"{RAW_DIR}/EchoNet-Dynamic"

# دانلود مستقیم از کاگل به حافظه محلی (بسیار سریع)
if not os.path.exists(dataset_path):
    print("📥 Downloading dataset directly to local memory...")
    dl_path = kagglehub.dataset_download("mahnurrahman/echonet-dynamic")
    shutil.copytree(dl_path, dataset_path, dirs_exist_ok=True)

# استخراج فریم‌ها و ساخت ماسک (چون روی حافظه محلی هستیم، با سرعت بالا انجام میشه)
if len(glob.glob(f"{IMG_DIR}/*.png")) < 1000:
    print("⚙️ Processing videos and generating masks locally...")
    df = pd.read_csv(f"{dataset_path}/EchoNet-Dynamic/VolumeTracings.csv")
    grouped = df.groupby(["FileName", "Frame"])

    SAVE_SIZE = 256
    counter = 0

    for (fname, frame_num), rows in tqdm(grouped):
        v_path = f"{dataset_path}/EchoNet-Dynamic/Videos/{fname}"
        if not os.path.exists(v_path): continue

        cap = cv2.VideoCapture(v_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        cap.release()

        if not ret: continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        mask = np.zeros(frame.shape, dtype=np.uint8)

        p1, p2 = [], []
        for _, r in rows.iterrows():
            p1.append([int(r["X1"]), int(r["Y1"])])
            p2.append([int(r["X2"]), int(r["Y2"])])

        points = np.array(p1 + p2[::-1], dtype=np.int32)
        if len(points) > 5:
            cv2.fillPoly(mask, [points], 255)
            frame = cv2.resize(frame, (SAVE_SIZE, SAVE_SIZE))
            mask = cv2.resize(mask, (SAVE_SIZE, SAVE_SIZE))

            cv2.imwrite(f"{IMG_DIR}/{counter}.png", frame)
            cv2.imwrite(f"{MASK_DIR}/{counter}.png", mask)
            counter += 1

print(f"✅ Total images generated: {len(glob.glob(f'{IMG_DIR}/*.png'))}")
print("🚀 Dataset is ready! You can start training now.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.1 MB/s eta 0:00:00
✅ Paths ready. Models go to Drive, Data stays Local (Ultra Fast).
📥 Downloading dataset directly to local memory...


100%|██████████| 6.56G/6.56G [01:12<00:00, 97.4MB/s]

Extracting files...


⚙️ Processing videos and generating masks locally...


100%|██████████| 20050/20050 [02:32<00:00, 131.08it/s]


✅ Total images generated: 20048
🚀 Dataset is ready! You can start training now.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

images_list = sorted(os.listdir(IMG_DIR))
# random_state=42 ensures the train/val split is exactly the same across different Colab accounts!
train_imgs, valid_imgs = train_test_split(images_list, test_size=0.1, random_state=42)

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ElasticTransform(p=0.2),
    A.GridDistortion(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])
valid_aug = A.Compose([A.Normalize(mean=(0.5,), std=(0.5,)), ToTensorV2()])

class EchoData(Dataset):
    def __init__(self, imgs, transform):
        self.imgs = imgs
        self.transform = transform

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        name = self.imgs[idx]
        img = cv2.imread(f"{IMG_DIR}/{name}", cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(f"{MASK_DIR}/{name}", cv2.IMREAD_GRAYSCALE)
        aug = self.transform(image=img, mask=mask)
        return aug["image"].float(), (aug["mask"].unsqueeze(0).float() / 255.0)

train_loader = DataLoader(EchoData(train_imgs, train_aug), batch_size=16, shuffle=True, num_workers=2)
valid_loader = DataLoader(EchoData(valid_imgs, valid_aug), batch_size=16, shuffle=False, num_workers=2)


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.UnetPlusPlus(
    encoder_name="timm-efficientnet-b4",
    encoder_weights="noisy-student",
    in_channels=1,
    classes=1
).to(device)

dice_loss = smp.losses.DiceLoss(mode="binary")
bce_loss = nn.BCEWithLogitsLoss()
def criterion(pred, target): return 0.5 * dice_loss(pred, target) + 0.5 * bce_loss(pred, target)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
scaler = torch.amp.GradScaler('cuda')
print("deviceeee:" , device)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

deviceeee: cuda


In [ ]:
LATEST_CKPT = f"{CKPT_DIR}/latest.pth"
BEST_CKPT = f"{CKPT_DIR}/best.pth"

start_epoch, best_dice, patience = 0, 0.0, 0
MAX_EPOCHS, EARLY_STOP = 100, 10

# ---- AUTO RESUME LOGIC ----
if os.path.exists(LATEST_CKPT):
    print("🔄 Existing checkpoint found! Resuming training...")
    ckpt = torch.load(LATEST_CKPT, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = ckpt['epoch']
    best_dice = ckpt['best_dice']
    patience = ckpt['patience']
    print(f"✅ Resumed successfully from Epoch {start_epoch} | Best Dice so far: {best_dice:.4f}")
else:
    print("🚀 Starting training from scratch...")
# ---------------------------

def calc_dice(p, t):
    p = (p > 0.5).float()
    return (2*(p*t).sum() + 1e-6) / (p.sum() + t.sum() + 1e-6)

for epoch in range(start_epoch, MAX_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{MAX_EPOCHS} ---")

    model.train()
    train_loss = 0
    for imgs, masks in tqdm(train_loader, desc="Train"):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            preds = model(imgs)
            loss = criterion(preds, masks)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    model.eval()
    val_loss, val_dice = 0, 0
    with torch.no_grad():
        for imgs, masks in tqdm(valid_loader, desc="Valid"):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            val_dice += calc_dice(torch.sigmoid(preds), masks).item()

    val_dice /= len(valid_loader)
    scheduler.step(val_dice)
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Val Dice: {val_dice:.4f}")

    if val_dice > best_dice:
        best_dice, patience = val_dice, 0
        torch.save(model.state_dict(), BEST_CKPT)
        print("⭐ New Best Model Saved!")
    else:
        patience += 1

    # Save checkpoint immediately to Google Drive (allows safe switching)
    torch.save({
        'epoch': epoch + 1,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'best_dice': best_dice,
        'patience': patience
    }, LATEST_CKPT)

    if patience >= EARLY_STOP:
        print("🛑 Early Stopping triggered. Training finished.")
        break


🔄 Existing checkpoint found! Resuming training...
✅ Resumed successfully from Epoch 19 | Best Dice so far: 0.9207

--- Epoch 20/100 ---


Valid: 100%|██████████| 126/126 [00:23<00:00,  5.38it/s]


Train Loss: 0.0489 | Val Dice: 0.9200

--- Epoch 21/100 ---


Valid: 100%|██████████| 126/126 [00:23<00:00,  5.40it/s]


Train Loss: 0.0480 | Val Dice: 0.9202

--- Epoch 22/100 ---


...

Train Loss: 0.0455 | Val Dice: 0.9203

--- Epoch 26/100 ---


Valid: 100%|██████████| 126/126 [00:23<00:00,  5.41it/s]


Train Loss: 0.0448 | Val Dice: 0.9203

--- Epoch 27/100 ---


Valid: 100%|██████████| 126/126 [00:23<00:00,  5.39it/s]


Train Loss: 0.0447 | Val Dice: 0.9202
🛑 Early Stopping triggered. Training finished.


In [ ]:
# ==========================================
# 1. SETUP & PATHS (Drive for Models, Local for Data)
# ==========================================
from google.colab import drive
import os
import kagglehub
import shutil
import cv2
import pandas as pd
import numpy as np
import glob
from tqdm import tqdm

# اتصال درایو فقط برای ذخیره فایل مدل (.pth)
drive.mount('/content/drive')

!pip install -q kagglehub segmentation-models-pytorch albumentations timm

# مسیر ذخیره مدل‌ها در درایو (تا با بسته شدن کولب پاک نشوند)
CKPT_DIR = "/content/drive/MyDrive/EchoNet_Colab/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# 🚀 مسیرهای محلی پرسرعت روی خود سیستم کولب
LOCAL_DIR = "/content/local_data"
RAW_DIR = f"{LOCAL_DIR}/raw"
IMG_DIR = f"{LOCAL_DIR}/images"
MASK_DIR = f"{LOCAL_DIR}/masks"

for d in [RAW_DIR, IMG_DIR, MASK_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Paths ready. Models go to Drive, Data stays Local (Ultra Fast).")

# ==========================================
# 2. DOWNLOAD & GENERATE DATASET LOCALLY
# ==========================================
dataset_path = f"{RAW_DIR}/EchoNet-Dynamic"

# دانلود مستقیم از کاگل به حافظه محلی (بسیار سریع)
if not os.path.exists(dataset_path):
    print("📥 Downloading dataset directly to local memory...")
    dl_path = kagglehub.dataset_download("mahnurrahman/echonet-dynamic")
    shutil.copytree(dl_path, dataset_path, dirs_exist_ok=True)

# استخراج فریم‌ها و ساخت ماسک (چون روی حافظه محلی هستیم، با سرعت بالا انجام میشه)
if len(glob.glob(f"{IMG_DIR}/*.png")) < 1000:
    print("⚙️ Processing videos and generating masks locally...")
    df = pd.read_csv(f"{dataset_path}/EchoNet-Dynamic/VolumeTracings.csv")
    grouped = df.groupby(["FileName", "Frame"])

    SAVE_SIZE = 256
    counter = 0

    for (fname, frame_num), rows in tqdm(grouped):
        v_path = f"{dataset_path}/EchoNet-Dynamic/Videos/{fname}"
        if not os.path.exists(v_path): continue

        cap = cv2.VideoCapture(v_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        cap.release()

        if not ret: continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        mask = np.zeros(frame.shape, dtype=np.uint8)

        p1, p2 = [], []
        for _, r in rows.iterrows():
            p1.append([int(r["X1"]), int(r["Y1"])])
            p2.append([int(r["X2"]), int(r["Y2"])])

        points = np.array(p1 + p2[::-1], dtype=np.int32)
        if len(points) > 5:
            cv2.fillPoly(mask, [points], 255)
            frame = cv2.resize(
                frame,
                (SAVE_SIZE, SAVE_SIZE),
                interpolation=cv2.INTER_AREA
            )

            mask = cv2.resize(
                mask,
                (SAVE_SIZE, SAVE_SIZE),
                interpolation=cv2.INTER_NEAREST
            )

            cv2.imwrite(f"{IMG_DIR}/{counter}.png", frame)
            cv2.imwrite(f"{MASK_DIR}/{counter}.png", mask)
            counter += 1

print(f"✅ Total images generated: {len(glob.glob(f'{IMG_DIR}/*.png'))}")
print("🚀 Dataset is ready! You can start training now.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Paths ready. Models go to Drive, Data stays Local (Ultra Fast).
✅ Total images generated: 20048
🚀 Dataset is ready! You can start training now.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

images_list = sorted(os.listdir(IMG_DIR))

train_imgs, valid_imgs = train_test_split(
    images_list,
    test_size=0.1,
    random_state=42
)

# ==========================================
# BETTER AUGMENTATION
# ==========================================
train_aug = A.Compose([

    A.HorizontalFlip(p=0.5),

    A.ShiftScaleRotate(
        shift_limit=0.08,
        scale_limit=0.15,
        rotate_limit=20,
        border_mode=cv2.BORDER_CONSTANT,
        p=0.7
    ),

    A.ElasticTransform(
        alpha=20,
        sigma=7,
        p=0.2
    ),

    A.GridDistortion(p=0.2),

    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.5
    ),

    A.GaussNoise(p=0.2),

    A.Normalize(mean=(0.5,), std=(0.5,)),

    ToTensorV2()
])

valid_aug = A.Compose([
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])

class EchoData(Dataset):

    def __init__(self, imgs, transform):
        self.imgs = imgs
        self.transform = transform

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):

        name = self.imgs[idx]

        img = cv2.imread(
            f"{IMG_DIR}/{name}",
            cv2.IMREAD_GRAYSCALE
        )

        mask = cv2.imread(
            f"{MASK_DIR}/{name}",
            cv2.IMREAD_GRAYSCALE
        )

        aug = self.transform(image=img, mask=mask)

        img = aug["image"].float()

        mask = (
            aug["mask"]
            .unsqueeze(0)
            .float() / 255.0
        )

        return img, mask

train_loader = DataLoader(
    EchoData(train_imgs, train_aug),
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    EchoData(valid_imgs, valid_aug),
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("✅ Data loaders ready.")

✅ Data loaders ready.


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# SAME MODEL ARCHITECTURE AS OLD CHECKPOINT
# ==========================================
model = smp.UnetPlusPlus(
    encoder_name="timm-efficientnet-b4",
    encoder_weights="noisy-student",
    in_channels=1,
    classes=1
).to(device)

# ==========================================
# BETTER LOSS
# ==========================================
tversky_loss = smp.losses.TverskyLoss(
    mode='binary',
    alpha=0.3,
    beta=0.7
)

bce_loss = nn.BCEWithLogitsLoss()

def criterion(pred, target):
    return (
        0.7 * tversky_loss(pred, target)
        +
        0.3 * bce_loss(pred, target)
    )

# ==========================================
# LOWER LR FOR FINETUNING
# ==========================================
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ==========================================
# SCHEDULER
# ==========================================
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

# ==========================================
# AMP
# ==========================================
scaler = torch.amp.GradScaler('cuda')

print("✅ Fine-tuning model ready on:", device)

✅ Fine-tuning model ready on: cuda


In [ ]:
LATEST_CKPT = f"{CKPT_DIR}/latest_v2.pth"
BEST_CKPT = f"{CKPT_DIR}/best_v2.pth"

start_epoch = 0
best_dice = 0.9207
patience = 0

MAX_EPOCHS = 50
EARLY_STOP = 12

# ==========================================
# LOAD OLD BEST MODEL
# ==========================================
OLD_BEST = f"{CKPT_DIR}/best.pth"

if os.path.exists(OLD_BEST):

    print("🔄 Loading previous best model...")

    model.load_state_dict(
        torch.load(OLD_BEST, map_location=device)
    )

    print("✅ Previous best model loaded.")

# ==========================================
# DICE FUNCTION
# ==========================================
def calc_dice(preds, targets, eps=1e-6):

    preds = (preds > 0.5).float()

    intersection = (
        preds * targets
    ).sum(dim=(1,2,3))

    union = (
        preds.sum(dim=(1,2,3))
        +
        targets.sum(dim=(1,2,3))
    )

    dice = (
        2.0 * intersection + eps
    ) / (union + eps)

    return dice.mean()

# ==========================================
# TRAIN LOOP
# ==========================================
for epoch in range(start_epoch, MAX_EPOCHS):

    print(f"\n🚀 Epoch {epoch+1}/{MAX_EPOCHS}")

    # ================= TRAIN =================
    model.train()

    train_loss = 0

    for imgs, masks in tqdm(train_loader, desc="Train"):

        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):

            preds = model(imgs)

            loss = criterion(preds, masks)

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    # ================= VALID =================
    model.eval()

    val_dice = 0

    with torch.no_grad():

        for imgs, masks in tqdm(valid_loader, desc="Valid"):

            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            preds = model(imgs)

            preds = torch.sigmoid(preds)

            val_dice += calc_dice(
                preds,
                masks
            ).item()

    val_dice /= len(valid_loader)

    scheduler.step(val_dice)

    current_lr = optimizer.param_groups[0]['lr']

    print(
        f"Train Loss: {train_loss/len(train_loader):.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"LR: {current_lr:.7f}"
    )

    # ==========================================
    # SAVE BEST MODEL
    # ==========================================
    if val_dice > best_dice:

        best_dice = val_dice
        patience = 0

        torch.save(
            model.state_dict(),
            BEST_CKPT
        )

        print("⭐ NEW BEST MODEL SAVED!")

    else:

        patience += 1

    # ==========================================
    # SAVE LATEST CHECKPOINT
    # ==========================================
    torch.save({

        'epoch': epoch + 1,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'best_dice': best_dice,
        'patience': patience

    }, LATEST_CKPT)

    print(f"📌 Best Dice So Far: {best_dice:.4f}")

    # ==========================================
    # EARLY STOPPING
    # ==========================================
    if patience >= EARLY_STOP:

        print("🛑 Early stopping triggered.")

        break

🔄 Loading previous best model...
✅ Previous best model loaded.

🚀 Epoch 1/50


Valid: 100%|██████████| 126/126 [00:22<00:00,  5.55it/s]


Train Loss: 0.0630 | Val Dice: 0.9138 | LR: 0.0001000
📌 Best Dice So Far: 0.9207

🚀 Epoch 2/50


Valid: 100%|██████████| 126/126 [00:22<00:00,  5.56it/s]


Train Loss: 0.0596 | Val Dice: 0.9138 | LR: 0.0001000
📌 Best Dice So Far: 0.9207

🚀 Epoch 3/50


...

Valid: 100%|██████████| 126/126 [00:22<00:00,  5.61it/s]


Train Loss: 0.0574 | Val Dice: 0.9130 | LR: 0.0001000
📌 Best Dice So Far: 0.9207

🚀 Epoch 6/50


Valid: 100%|██████████| 126/126 [00:22<00:00,  5.59it/s]


Train Loss: 0.0575 | Val Dice: 0.9111 | LR: 0.0000500
📌 Best Dice So Far: 0.9207

🚀 Epoch 7/50


Train:  38%|███▊      | 430/1128 [02:32<03:57,  2.94it/s]